In [2]:
pip install osmnx geopandas shapely geopy numpy pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
from shapely.geometry import Point
from shapely.ops import unary_union
from geopy.distance import geodesic
import numpy as np
import folium

# === PARÁMETROS ===
nombre_comuna = "Comuna 1 Norte, Bucaramanga, Colombia"
distancia_metros = 100        # distancia entre puntos candidatos
radio_cobertura = 50         # radio de cobertura de cada cesta (m)
max_iteraciones = 150         # límite para no generar demasiados puntos

# === 1️⃣ DESCARGAR O LEER GEOMETRÍA DE LA COMUNA ===
comuna = ox.geocode_to_gdf(nombre_comuna)
poligono_comuna = comuna.geometry.iloc[0]

# === 2️⃣ DESCARGAR VÍAS DE LA COMUNA ===
vias = ox.graph_from_polygon(poligono_comuna, network_type='drive')
vias_gdf = ox.graph_to_gdfs(vias, nodes=False, edges=True)

vias_gdf = vias_gdf.to_crs(epsg=3116)

# === 3️⃣ GENERAR PUNTOS CANDIDATOS SOBRE LAS VÍAS ===
def generar_puntos(vias_gdf, distancia):
    puntos = []
    for _, row in vias_gdf.iterrows():
        line = row.geometry
        if line is None:
            continue
        for d in np.arange(0, line.length, distancia):
            punto = line.interpolate(d)
            puntos.append(punto)
    return gpd.GeoDataFrame(geometry=puntos, crs=vias_gdf.crs)

puntos_candidatos = generar_puntos(vias_gdf, distancia_metros)
puntos_candidatos = puntos_candidatos.to_crs(epsg=4326)

In [5]:
center_lat = puntos_candidatos.geometry.y.mean()
center_lon = puntos_candidatos.geometry.x.mean()

m2 = folium.Map(location=[center_lat, center_lon], zoom_start=14)

# Añadir puntos candidatos
for _, row in puntos_candidatos.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color='green',
        fill=True,
        fill_opacity=0.0,
        popup='Punto candidato'
    ).add_to(m2)

m2.save("m2.html")
import webbrowser
webbrowser.open("m2.html")

True

In [6]:
df = pd.read_csv(r"D:\modelo\COORDENADAS\1norte.csv", sep=";")
df

,Latitud,Longitud
0,7.157456,-73.130836
1,7.158106,-73.131281
2,7.157773,-73.130848
3,7.157858,-73.130422
4,7.158234,-73.132073
...,...,...
129,7.162622,-73.140381
130,7.162557,-73.140199
131,7.162369,-73.140393
132,7.162555,-73.140400


In [ ]:
#!jupyter trust MALLA_PUNTOS.ipynb

usage: python.exe c:\Users\UIS\AppData\Roaming\Python\Python314\Scripts\jupyter
       [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir] [--paths]
       [--json] [--debug]
       [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand            the subcommand to launch

options:
  -h, --help            show this help message and exit
  --version             show the versions of core jupyter packages and exit
  --config-dir          show Jupyter config dir
  --data-dir            show Jupyter data dir
  --runtime-dir         show Jupyter runtime dir
  --paths               show all Jupyter paths. Add --json for machine-
                        readable format.
  --json                output paths as machine-readable json
  --debug               output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-trust` not found.


In [7]:
radio_cobertura = 50
gdf_existentes = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.Longitud, df.Latitud), # Correct order for x, y
    crs="EPSG:4326"
)

# === 5️⃣ CALCULAR ZONA DE COBERTURA ACTUAL ===
gdf_existentes_utm = gdf_existentes.to_crs(epsg=3116)
buffers = gdf_existentes_utm.buffer(radio_cobertura)
zona_cubierta = unary_union(buffers)

# === 6️⃣ EXPANSIÓN ITERATIVA (controlada por distancia límite) ===

puntos_utm = puntos_candidatos.to_crs(epsg=3116)
instaladas = []
iteracion = 0

# Nuevo parámetro: distancia máxima permitida (en metros)
distancia_maxima = 1000  # puedes ajustarla según la densidad deseada

while True:
    # Puntos fuera del área cubierta actual
    fuera = puntos_utm[~puntos_utm.within(zona_cubierta)]
    if len(fuera) == 0:
        print("✅ No quedan puntos fuera del área cubierta.")
        break

    # Calcular distancia mínima de cada punto al área cubierta
    distancias = fuera.distance(zona_cubierta)

    # Seleccionar el más cercano al borde actual
    idx_min = distancias.idxmin()
    dist_min = distancias.min()

    # Si el más cercano está demasiado lejos → detener expansión
    if dist_min > distancia_maxima:
        print(f"🛑 Detenido: el punto más cercano ({dist_min:.2f} m) supera el límite de {distancia_maxima} m.")
        break

    # Agregar nueva cesta
    nueva = fuera.loc[[idx_min]]
    # Replace deprecated unary_union attribute with union_all() method
    instaladas.append(nueva)
    zona_cubierta = zona_cubierta.union(nueva.buffer(radio_cobertura).union_all())

    iteracion += 1
    print(f"🪣 Nueva cesta #{iteracion} instalada (a {dist_min:.1f} m del borde). Cobertura ampliada.")

# === 7️⃣ RESULTADOS ===
if len(instaladas) > 0:
    nuevas_cestas = gpd.GeoDataFrame(pd.concat(instaladas, ignore_index=True), crs=puntos_utm.crs).to_crs(epsg=4326)
else:
    nuevas_cestas = gpd.GeoDataFrame(columns=['geometry'], crs=puntos_utm.crs).to_crs(epsg=4326)

print("\n=== RESULTADOS ===")
print(f"Total de cestas nuevas: {len(nuevas_cestas)}")
print(f"Total de iteraciones ejecutadas: {iteracion}")


# === 7️⃣ RESULTADOS ===
nuevas_cestas = gpd.GeoDataFrame(pd.concat(instaladas, ignore_index=True), crs=puntos_utm.crs).to_crs(epsg=4326)
print("\n=== RESULTADOS ===")
print(f"Total de cestas nuevas: {len(nuevas_cestas)}")
print(f"Total de iteraciones ejecutadas: {iteracion}")

# Puedes mostrar las ubicaciones en un mapa (opcional con folium)
import folium
m = folium.Map(location=[gdf_existentes.geometry.y.mean(), gdf_existentes.geometry.x.mean()], zoom_start=14)
for _, row in gdf_existentes.iterrows():
    folium.CircleMarker(location=[row.geometry.y, row.geometry.x], color='blue', radius=4, popup='Existente').add_to(m)
for _, row in nuevas_cestas.iterrows():
    folium.CircleMarker(location=[row.geometry.y, row.geometry.x], color='red', radius=4, popup='Nueva').add_to(m)

m.save("mapa.html")
import webbrowser
webbrowser.open("mapa.html")

🪣 Nueva cesta #1 instalada (a 1.1 m del borde). Cobertura ampliada.
🪣 Nueva cesta #2 instalada (a 1.5 m del borde). Cobertura ampliada.
🪣 Nueva cesta #3 instalada (a 2.0 m del borde). Cobertura ampliada.
🪣 Nueva cesta #4 instalada (a 2.7 m del borde). Cobertura ampliada.
🪣 Nueva cesta #5 instalada (a 2.8 m del borde). Cobertura ampliada.
🪣 Nueva cesta #6 instalada (a 3.3 m del borde). Cobertura ampliada.
🪣 Nueva cesta #7 instalada (a 2.5 m del borde). Cobertura ampliada.
🪣 Nueva cesta #8 instalada (a 3.5 m del borde). Cobertura ampliada.
🪣 Nueva cesta #9 instalada (a 4.4 m del borde). Cobertura ampliada.
🪣 Nueva cesta #10 instalada (a 4.7 m del borde). Cobertura ampliada.
🪣 Nueva cesta #11 instalada (a 5.2 m del borde). Cobertura ampliada.
🪣 Nueva cesta #12 instalada (a 6.3 m del borde). Cobertura ampliada.
🪣 Nueva cesta #13 instalada (a 0.9 m del borde). Cobertura ampliada.
🪣 Nueva cesta #14 instalada (a 1.8 m del borde). Cobertura ampliada.
🪣 Nueva cesta #15 instalada (a 1.6 m del bo

True

In [8]:
from geopy.distance import geodesic
import pandas as pd
import os

# Parámetro: distancia máxima para conectar puntos (en metros)
distancia_max_conexion = 100  # ajusta según la densidad deseada

# Asegurar que las cestas estén numeradas de 1 a n
nuevas_cestas = nuevas_cestas.reset_index(drop=True)
nuevas_cestas["id"] = nuevas_cestas.index + 1

# Mostrar número total de puntos (nodos)
print(f"\n📍 Total de nuevas cestas (nodos): {len(nuevas_cestas)}")

# Lista para guardar los pares conectados
conexiones = []

# Secuencia incremental: cada nueva cesta se conecta con la más cercana anterior dentro del rango permitido
for i in range(len(nuevas_cestas)):
    if i == 0:
        continue  # la primera no tiene anterior
    p_actual = nuevas_cestas.loc[i, "geometry"]
    anteriores = nuevas_cestas.iloc[:i]

    # Calcular distancia a cada una de las anteriores
    distancias = anteriores["geometry"].apply(
        lambda g: geodesic((p_actual.y, p_actual.x), (g.y, g.x)).meters
    )

    # Buscar la más cercana dentro del rango máximo
    idx_min = distancias.idxmin()
    dist_min = distancias.min()

    if dist_min <= distancia_max_conexion:
        origen_geom = nuevas_cestas.loc[idx_min, "geometry"]
        destino_geom = nuevas_cestas.loc[i, "geometry"]
        conexiones.append({
            "i": idx_min + 1,
            "j": i + 1,
            "Dij": round(dist_min, 2),
            "lat_i": origen_geom.y,
            "lon_i": origen_geom.x,
            "lat_j": destino_geom.y,
            "lon_j": destino_geom.x
        })
    else:
        print(f"⚠️ Cesta #{i+1} está aislada ({dist_min:.1f} m de la más cercana, fuera de rango).")


📍 Total de nuevas cestas (nodos): 329
⚠️ Cesta #2 está aislada (1325.0 m de la más cercana, fuera de rango).
⚠️ Cesta #4 está aislada (1279.5 m de la más cercana, fuera de rango).
⚠️ Cesta #5 está aislada (201.3 m de la más cercana, fuera de rango).
⚠️ Cesta #6 está aislada (422.5 m de la más cercana, fuera de rango).
⚠️ Cesta #12 está aislada (193.3 m de la más cercana, fuera de rango).
⚠️ Cesta #21 está aislada (104.9 m de la más cercana, fuera de rango).
⚠️ Cesta #25 está aislada (227.8 m de la más cercana, fuera de rango).
⚠️ Cesta #72 está aislada (112.2 m de la más cercana, fuera de rango).
⚠️ Cesta #98 está aislada (330.1 m de la más cercana, fuera de rango).
⚠️ Cesta #105 está aislada (129.2 m de la más cercana, fuera de rango).
⚠️ Cesta #110 está aislada (143.0 m de la más cercana, fuera de rango).
⚠️ Cesta #114 está aislada (157.2 m de la más cercana, fuera de rango).
⚠️ Cesta #192 está aislada (162.7 m de la más cercana, fuera de rango).
⚠️ Cesta #193 está aislada (154.0 m 

In [9]:
df_conexiones = pd.DataFrame(conexiones).sort_values(by=["i"]).reset_index(drop=True)

print("\n=== CONEXIONES SECUENCIALES ENTRE NUEVAS CESTAS ===")
print(df_conexiones)
print(f"\n🔗 Total de conexiones registradas: {len(df_conexiones)}")

# Mostrar total de filas (nodos conectados)
print(f"📊 Total de filas en df_conexiones: {df_conexiones.shape[0]}")


=== CONEXIONES SECUENCIALES ENTRE NUEVAS CESTAS ===
       i    j    Dij     lat_i      lon_i     lat_j      lon_j
0      1    8  90.41  7.149447 -73.135557  7.149837 -73.136276
1      1  106  73.21  7.149447 -73.135557  7.148810 -73.135375
2      1   23  57.47  7.149447 -73.135557  7.149888 -73.135283
3      2   76  65.13  7.151723 -73.147335  7.151225 -73.147021
4      2    3  95.35  7.151723 -73.147335  7.151993 -73.146515
..   ...  ...    ...       ...        ...       ...        ...
306  318  319  77.32  7.150101 -73.142391  7.150800 -73.142408
307  319  320  53.57  7.150800 -73.142408  7.151029 -73.141980
308  319  321  80.90  7.150800 -73.142408  7.150847 -73.143138
309  320  324  85.84  7.151029 -73.141980  7.151645 -73.141507
310  321  322  53.54  7.150847 -73.143138  7.150363 -73.143129

[311 rows x 7 columns]

🔗 Total de conexiones registradas: 311
📊 Total de filas en df_conexiones: 311


In [10]:
carpeta_salida = r"D:\modelo\DISTANCIAS"
os.makedirs(carpeta_salida, exist_ok=True)

ruta_salida = f"{carpeta_salida}\\1norte.csv"
df_conexiones.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

print(f"💾 Archivo guardado en: {ruta_salida}")

💾 Archivo guardado en: D:\modelo\DISTANCIAS\1norte.csv


In [11]:
m = folium.Map(location=[gdf_existentes.geometry.y.mean(), gdf_existentes.geometry.x.mean()], zoom_start=14)

# Marcar cestas existentes (azul)
for _, row in gdf_existentes.iterrows():
    folium.CircleMarker(location=[row.geometry.y, row.geometry.x],
                        color='blue', radius=4, popup='Existente').add_to(m)

# Marcar nuevas cestas (rojo)
for _, row in nuevas_cestas.iterrows():
    folium.CircleMarker(location=[row.geometry.y, row.geometry.x],
                        color='red', radius=4, popup=f"Nueva cesta #{row['id']}").add_to(m)

# Dibujar líneas de conexión (verde)
for _, row in df_conexiones.iterrows():
    origen = nuevas_cestas.loc[row["i"] - 1, "geometry"]
    destino = nuevas_cestas.loc[row["j"] - 1, "geometry"]
    folium.PolyLine(
        locations=[(origen.y, origen.x), (destino.y, destino.x)],
        color='green',
        weight=2,
        opacity=0.8
    ).add_to(m)

m.save("mapa.html")
import webbrowser
webbrowser.open("mapa.html")

True